<a href="https://colab.research.google.com/github/ketanp23/deeplearningclass/blob/main/DL_Lab4_Backpropagation_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DL Lab 4 · Back-Propagation from Scratch

> **Deep Learning foundations — Lab.** Run top to bottom (Runtime → Run all). Read the notes, run the code, and finish
> the **Your turn 🧪** cell. We build from scratch with NumPy so nothing is hidden.

## What you'll build
Back-prop demystified: it's just the **chain rule** applied layer by layer. You'll implement forward and backward
passes for a 2-layer network and **verify every gradient** against a numerical estimate.

In [1]:
import numpy as np
np.random.seed(0)
def sigmoid(z): return 1/(1+np.exp(-z))

## 1 · The chain rule, by hand
For `x → h = σ(w1·x+b1) → y = σ(w2·h+b2)`, loss `L = ½(y−t)²`:
$$\frac{\partial L}{\partial w_2}=(y-t)\,y(1-y)\,h,\qquad
\frac{\partial L}{\partial w_1}=(y-t)\,y(1-y)\,w_2\,h(1-h)\,x$$

In [2]:
x, t = 1.0, 1.0
w1, b1, w2, b2 = 0.8, -0.3, 1.1, 0.2

# forward
h = sigmoid(w1*x + b1)
y = sigmoid(w2*h + b2)
L = 0.5*(y-t)**2

# backward (chain rule)
dy  = (y - t) * y*(1-y)
gw2 = dy * h;              gb2 = dy
dh  = dy * w2 * h*(1-h)
gw1 = dh * x;              gb1 = dh
print(f"forward: h={h:.4f}  y={y:.4f}  loss={L:.5f}")
print(f"grads:   w1={gw1:.5f}  w2={gw2:.5f}")

forward: h=0.6225  y=0.7078  loss=0.04269
grads:   w1=-0.01562  w2=-0.03762


## 2 · Gradient check (trust, but verify)

In [3]:
def loss_of(w1,b1,w2,b2):
    h=sigmoid(w1*x+b1); y=sigmoid(w2*h+b2); return 0.5*(y-t)**2
eps=1e-6
num_w2=(loss_of(w1,b1,w2+eps,b2)-loss_of(w1,b1,w2-eps,b2))/(2*eps)
num_w1=(loss_of(w1+eps,b1,w2,b2)-loss_of(w1-eps,b1,w2,b2))/(2*eps)
print(f"w2  analytic {gw2:.6f}  vs numeric {num_w2:.6f}")
print(f"w1  analytic {gw1:.6f}  vs numeric {num_w1:.6f}")
print("They match -> our back-prop is correct.")

w2  analytic -0.037618  vs numeric -0.037618
w1  analytic -0.015622  vs numeric -0.015622
They match -> our back-prop is correct.


## 3 · Vectorized back-prop for a whole layer
The same idea with matrices, trained on a tiny dataset.

In [4]:
# tiny 2 -> 3 -> 1 network on a toy AND-ish task
Xd = np.array([[0,0],[0,1],[1,0],[1,1]], float)
td = np.array([[0],[0],[0],[1]], float)      # logical AND
W1=np.random.randn(2,3); B1=np.zeros((1,3)); W2=np.random.randn(3,1); B2=np.zeros((1,1))
for it in range(4000):
    H = sigmoid(Xd@W1+B1); Y = sigmoid(H@W2+B2)          # forward
    dY = (Y-td)*Y*(1-Y)                                   # backward
    dW2 = H.T@dY; dB2 = dY.sum(0,keepdims=True)
    dH = (dY@W2.T)*H*(1-H)
    dW1 = Xd.T@dH; dB1 = dH.sum(0,keepdims=True)
    for p,g in [(W1,dW1),(B1,dB1),(W2,dW2),(B2,dB2)]: p -= 0.5*g/len(Xd)
print("AND predictions:", np.round(Y.ravel(),2), " target:", td.ravel())

AND predictions: [0.   0.05 0.04 0.94]  target: [0. 0. 0. 1.]


## 4 · Why back-prop is efficient
A naive approach recomputes shared terms for every weight. Back-prop computes each intermediate gradient **once** and
reuses it — so one backward pass yields the gradient for *all* parameters. That reuse is what makes training big
networks feasible.

## Your turn 🧪
1. Add a second hidden layer (2 → 3 → 3 → 1) and extend the backward pass. Gradient-check it.
2. Swap the hidden activation to ReLU; its local gradient is `(H>0)`. Re-derive `dH`.
3. Break back-prop on purpose: drop the `*H*(1-H)` term in `dH` and watch training stall — that term is the chain rule doing its job.

In [5]:
# Your turn: ReLU hidden layer — fill in the derivative
def relu(z): return np.maximum(0,z)
# H = relu(Xd@W1+B1);  dH = (dY@W2.T) * (H>0)   # <- ReLU's local gradient